<a href="https://colab.research.google.com/github/mebre7/house-price-predictor-mlops/blob/colab_branch/notebooks/01_eda_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA)
Exploratory Data Analysis (EDA) is a crucial step in the data analysis process that involves examining and visualizing data to uncover patterns, relationships, and insights. It helps in understanding the underlying structure of the data, identifying anomalies, and informing subsequent modeling decisions.

## 1. Load Data from PostgreSQL which is inside Supabase

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus
from google.colab import userdata

ModuleNotFoundError: No module named 'google.colab'

In [4]:
DB_USER = userdata.get("SUPABASE_USER")
DB_PASSWORD = quote_plus(userdata.get("SUPABASE_PASSWORD"))
DB_HOST = userdata.get("SUPABASE_HOST")
DB_PORT = userdata.get("SUPABASE_PORT")
DB_NAME = userdata.get("SUPABASE_NAME")

# 2. Build SQLAlchemy connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
with engine.connect() as conn:
    df = pd.read_sql(
        text("SELECT * FROM housing_data"),
        conn
    )

df.head()

,Order,PID,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,None,IR1,Lvl,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,None,Reg,Lvl,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,None,IR1,Lvl,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,None,Reg,Lvl,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,None,IR1,Lvl,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


### Import packages

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

## 2. Basic Shape & Types Audit

In [5]:
df.shape

(2930, 82)

-> $2930$ rows and $82$ features/columns

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS_SubClass      2930 non-null   int64  
 3   MS_Zoning        2930 non-null   object 
 4   Lot_Frontage     2440 non-null   float64
 5   Lot_Area         2930 non-null   int64  
 6   Street           2930 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot_Shape        2930 non-null   object 
 9   Land_Contour     2930 non-null   object 
 10  Utilities        2930 non-null   object 
 11  Lot_Config       2930 non-null   object 
 12  Land_Slope       2930 non-null   object 
 13  Neighborhood     2930 non-null   object 
 14  Condition_1      2930 non-null   object 
 15  Condition_2      2930 non-null   object 
 16  Bldg_Type        2930 non-null   object 
 17  House_Style   

Of 82 columns:
- Float: 11
- Integer: 28
- Object (string): 43

In [7]:
df.dtypes

,0
Order,int64
PID,int64
MS_SubClass,int64
MS_Zoning,object
Lot_Frontage,float64
...,...
Mo_Sold,int64
Yr_Sold,int64
Sale_Type,object
Sale_Condition,object


In [18]:
df.describe()

,Order,PID,MS_SubClass,Lot_Frontage,Lot_Area,Overall_Qual,Overall_Cond,Year_Built,Year_Remod/Add,Mas_Vnr_Area,...,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,3Ssn_Porch,Screen_Porch,Pool_Area,Misc_Val,Mo_Sold,Yr_Sold,SalePrice
count,2930.00000,2.930000e+03,2930.000000,2440.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2907.000000,...,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000
mean,1465.50000,7.144645e+08,57.387372,69.224590,10147.921843,6.094881,5.563140,1971.356314,1984.266553,101.896801,...,93.751877,47.533447,23.011604,2.592491,16.002048,2.243345,50.635154,6.216041,2007.790444,180796.060068
std,845.96247,1.887308e+08,42.638025,23.365335,7880.017759,1.411026,1.111537,30.245361,20.860286,179.112611,...,126.361562,67.483400,64.139059,25.141331,56.087370,35.597181,566.344288,2.714492,1.316613,79886.692357
min,1.00000,5.263011e+08,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,12789.000000
25%,733.25000,5.284770e+08,20.000000,58.000000,7440.250000,5.000000,5.000000,1954.000000,1965.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000,129500.000000
50%,1465.50000,5.354536e+08,50.000000,68.000000,9436.500000,6.000000,5.000000,1973.000000,1993.000000,0.000000,...,0.000000,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,160000.000000
75%,2197.75000,9.071811e+08,70.000000,80.000000,11555.250000,7.000000,6.000000,2001.000000,2004.000000,164.000000,...,168.000000,70.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,213500.000000
max,2930.00000,1.007100e+09,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,...,1424.000000,742.000000,1012.000000,508.000000,576.000000,800.000000,17000.000000,12.000000,2010.000000,755000.000000


### Numerical and Categorical Columns

In [19]:
# define numerical & categorical columns
numerical_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

In [20]:
# print columns
print(f'{len(numerical_features)} numerical features: {numerical_features}')
print(f'\n{len(categorical_features)} categorical features: {categorical_features}')

39 numerical features: ['Order', 'PID', 'MS_SubClass', 'Lot_Frontage', 'Lot_Area', 'Overall_Qual', 'Overall_Cond', 'Year_Built', 'Year_Remod/Add', 'Mas_Vnr_Area', 'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF', '1st_Flr_SF', '2nd_Flr_SF', 'Low_Qual_Fin_SF', 'Gr_Liv_Area', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Yr_Blt', 'Garage_Cars', 'Garage_Area', 'Wood_Deck_SF', 'Open_Porch_SF', 'Enclosed_Porch', '3Ssn_Porch', 'Screen_Porch', 'Pool_Area', 'Misc_Val', 'Mo_Sold', 'Yr_Sold', 'SalePrice']

43 categorical features: ['MS_Zoning', 'Street', 'Alley', 'Lot_Shape', 'Land_Contour', 'Utilities', 'Lot_Config', 'Land_Slope', 'Neighborhood', 'Condition_1', 'Condition_2', 'Bldg_Type', 'House_Style', 'Roof_Style', 'Roof_Matl', 'Exterior_1st', 'Exterior_2nd', 'Mas_Vnr_Type', 'Exter_Qual', 'Exter_Cond', 'Foundation', 'Bsmt_Qual', 'Bsmt_Cond', 'Bsmt_Exposure', 'BsmtFin_Type_1', 'BsmtFin_Type_2

There are:
- **39** numeric features
- **43** categorical features

## 3. Missing Value Analysis

In [21]:
df.isnull().sum()

,0
Order,0
PID,0
MS_SubClass,0
MS_Zoning,0
Lot_Frontage,490
...,...
Mo_Sold,0
Yr_Sold,0
Sale_Type,0
Sale_Condition,0


#### Frequency of each column
- Proportion of individual values in each column

In [25]:
for col in categorical_features:
  print(df[col].value_counts(normalize=True) * 100)
  print(f"\n----------------------------------")

MS_Zoning
RL         77.576792
RM         15.767918
FV          4.744027
RH          0.921502
C (all)     0.853242
I (all)     0.068259
A (agr)     0.068259
Name: proportion, dtype: float64

----------------------------------
Street
Pave    99.590444
Grvl     0.409556
Name: proportion, dtype: float64

----------------------------------
Alley
Grvl    60.606061
Pave    39.393939
Name: proportion, dtype: float64

----------------------------------
Lot_Shape
Reg    63.447099
IR1    33.412969
IR2     2.593857
IR3     0.546075
Name: proportion, dtype: float64

----------------------------------
Land_Contour
Lvl    89.863481
HLS     4.095563
Bnk     3.993174
Low     2.047782
Name: proportion, dtype: float64

----------------------------------
Utilities
AllPub    99.897611
NoSewr     0.068259
NoSeWa     0.034130
Name: proportion, dtype: float64

----------------------------------
Lot_Config
Inside     73.037543
Corner     17.440273
CulDSac     6.143345
FR2         2.901024
FR3         0.477816